# Tema 14 — Seguimiento de objetos (tracking) con YOLO y BoT-SORT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-07/Tema-14/Tema_14.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En este notebook usamos un detector **YOLO** junto con el algoritmo de seguimiento **BoT-SORT** para detectar personas en un video y **asignarles un identificador (ID) estable** que se mantiene cuadro a cuadro, incluso ante oclusiones.

> 💡 Si lo abres en Colab, ejecuta primero la celda **Setup para Google Colab** (instala `ultralytics`). En entornos locales esa celda no hace nada y puedes saltarla.

> ⚙️ El código **detecta el entorno automáticamente**: en **local** muestra el seguimiento en vivo con `cv2.imshow` (sal con la tecla `q`) y en **Colab** guarda el resultado en `salida_tracking.mp4` y lo reproduce en la última celda. Si no tienes el video `MOT17 04 FRCNN raw.mp4`, el notebook **descarga automáticamente** un video de peatones de ejemplo (`vtest.avi`), así que puedes ejecutarlo sin subir nada.

In [12]:
# === Setup para Google Colab ===
# Esta celda instala las dependencias que NO vienen preinstaladas en Colab.
# En entornos locales (VS Code / Jupyter) se ignora; si ya tienes ultralytics
# instalado también puedes saltarla.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # ultralytics -> YOLO + trackers (BoT-SORT). torch y opencv ya vienen en Colab.
    !pip install -q ultralytics
    print("Setup de Colab completado.")
else:
    print("Entorno local detectado, no se requiere setup adicional.")

Entorno local detectado, no se requiere setup adicional.


**Inicializar los parámetros y umbrales de seguimiento de objetos**

### ¿Qué hace este código?

Esto **no es código Python**, sino la **configuración del algoritmo de seguimiento BoT-SORT** en formato YAML. Por eso la celda empieza con la *magic* **`%%writefile custom_tracker.yaml`**, que **guarda todo el contenido en el archivo `custom_tracker.yaml`** (el mismo que carga la celda de tracking con `tracker="custom_tracker.yaml"`).

> ⚠️ Sin `%%writefile`, Jupyter intentaría ejecutar las líneas como Python y daría un error (`NameError: name 'botsort' is not defined`).

Cada parámetro controla cómo se asignan y conservan los IDs:

- **`track_high_thresh` / `track_low_thresh`**: umbrales de confianza para aceptar o recuperar detecciones.
- **`new_track_thresh`**: qué tan seguro debe estar el modelo para crear un **ID nuevo** (evita falsos positivos).
- **`track_buffer`**: cuántos cuadros "recuerda" a un objeto **oculto** antes de descartarlo (manejo de oclusiones).
- **`match_thresh`**: similitud necesaria para asociar una detección al mismo ID.
- **`with_reid` + `proximity_thresh` / `appearance_thresh`**: activan la **reidentificación visual**, comparando la apariencia para no cambiar de ID cuando un objeto reaparece.

In [13]:
%%writefile custom_tracker.yaml
# Ultralytics AGPL-3.0 License
# BoT-SORT tracker defaults for mode="track"

tracker_type: botsort # Define el uso de características avanzadas de BoT-SORT
track_high_thresh: 0.5 # Umbral inicial; valores altos limpian los rastros falsos
track_low_thresh: 0.1 # Umbral secundario para recuperar detecciones débiles
new_track_thresh: 0.7 # Exigencia para iniciar un ID nuevo (evita falsos positivos)
track_buffer: 250 # Cantidad de frames que "recuerda" un objeto oculto (Oclusión)
match_thresh: 0.8 # Similitud requerida para asociar el mismo ID
fuse_score: True # Fusiona la confianza de detección con el movimiento

# BoT-SORT specifics
gmc_method: none # 'sift' es preciso pero MUY lento en CPU; 'none' basta para cámara fija

# ReID model related thresh (Reidentificación visual)
proximity_thresh: 0.5 # Distancia máxima para considerar el ReID
appearance_thresh: 0.5 # Similitud visual requerida para no cambiar de ID
with_reid: False # Desactivado para acelerar; ponlo en True para reidentificación visual (más lento)
model: auto # Modelo automático para extraer características

Overwriting custom_tracker.yaml


**Seguimiento de objetos con asignación de cuadros delimitadores**

### ¿Qué hace este código?

Ejecuta el **bucle de seguimiento** sobre el video y funciona **tanto en local como en Colab**:

1. **Detecta el entorno** (`IN_COLAB`), porque `cv2.imshow` **no funciona en Colab**.
2. **Configura el dispositivo** eligiendo la mejor aceleración disponible (**CUDA** de NVIDIA → **MPS** de Apple Silicon → **CPU**) y carga el modelo **YOLO** (`yolo26x.pt`, la versión extra-grande, más precisa aunque más lenta).
3. Busca el video indicado y, **si no existe, descarga automáticamente un video de peatones de ejemplo** (`vtest.avi` de OpenCV); además prepara un **`cv2.VideoWriter`** para guardar el resultado en `salida_tracking.mp4`.
4. Por cada cuadro (*frame*) hace un **recorte central** de 300×300 px y llama a `model.track(...)` con `persist=True` para que el tracker (**BoT-SORT**, definido en `custom_tracker.yaml`) **mantenga los IDs**; se filtran solo personas (`classes=[0]`).
5. Dibuja el **bounding box**, un **color único por ID** y la etiqueta con ID + confianza, y **escribe el cuadro en el video de salida**.
6. **En local** además muestra la ventana en vivo (`cv2.imshow`, salir con **`q`**). **En Colab** solo guarda el archivo, que se visualiza en la siguiente celda.

> ⚡ **Nota de rendimiento:** el código **se adapta automáticamente al equipo de cada quien**: usa GPU si está disponible (**CUDA** en PC con tarjeta NVIDIA o **MPS** en Mac con Apple Silicon) y, si no, **CPU** (funciona igual, solo más lento). Se mantiene el modelo `yolo26x.pt` (más preciso pero lento, sobre todo en CPU) y, para acelerar sin cambiar de modelo, el tracker usa `gmc_method: none` y `with_reid: False`.


In [14]:
import os
import sys
import urllib.request
import cv2
import torch
from ultralytics import YOLO

# Detecta el entorno: en Colab no existe ventana gráfica (cv2.imshow no funciona).
IN_COLAB = "google.colab" in sys.modules

# 1. Configuración de Dispositivo y Modelo
# Selecciona la mejor aceleración disponible: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU.
if torch.cuda.is_available():
    device = 'cuda'
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

# Modelo: 'x' (extra-grande) es más preciso aunque más lento; 'n' (nano) sería más rápido.
model = YOLO('yolo26x.pt').to(device)

# 2. Configuración de Video (entrada y salida)
video_path = 'MOT17 04 FRCNN raw.mp4'   # tu video; si no existe, se descarga uno de ejemplo
output_path = 'salida_tracking.mp4'     # video anotado que se genera (sirve en local y Colab)

# Si no tienes el video, descargamos uno de peatones de ejemplo (vtest.avi de OpenCV).
if not os.path.exists(video_path):
    sample_path = 'vtest.avi'
    sample_url = 'https://github.com/opencv/opencv/raw/master/samples/data/vtest.avi'
    if not os.path.exists(sample_path):
        print(f"No se encontró '{video_path}'. Descargando video de ejemplo '{sample_path}'...")
        urllib.request.urlretrieve(sample_url, sample_path)
        print("Descarga completada.")
    video_path = sample_path

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video '{video_path}'.")

# Configuraciones de detección y tracking
SCORE_THRESH = 0.7
IOU_THRESH = 0.6
TARGET_CLASSES = [0]  # Solo personas

# 3. Parámetros del recorte central y del escritor de video de salida
crop_w, crop_h = 300, 300
fps = cap.get(cv2.CAP_PROP_FPS) or 25
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (crop_w, crop_h))

print(f"Iniciando Tracking en {device} ({'Colab' if IN_COLAB else 'local'}) con '{video_path}'...")
if not IN_COLAB:
    print("Presiona 'q' en la ventana para salir.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 4. Recorte (Crop) Central para optimizar el procesamiento
    h_orig, w_orig = frame.shape[:2]
    start_x = max(0, w_orig // 2 - (crop_w // 2))
    start_y = max(0, h_orig // 2 - (crop_h // 2))
    frame_crop = frame[start_y:start_y+crop_h, start_x:start_x+crop_w]
    # Aseguramos el tamaño exacto del recorte (por si el frame es más pequeño)
    frame_crop = cv2.resize(frame_crop, (crop_w, crop_h))

    # 5. Tracking
    results = model.track(
        source=frame_crop,
        persist=True,
        iou=IOU_THRESH,      # Umbral de Intersection over Union para asociación
        imgsz=416,          # Resolución de inferencia (múltiplo de 32 requerido por YOLO)
        classes=TARGET_CLASSES, # Filtrar solo personas
        tracker="custom_tracker.yaml", # Archivo de configuración del algoritmo de tracking
        device=device,
        verbose=False,
    )[0]

    # 6. Proyección de resultados si hay identificadores confirmados
    if results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy().astype(int)
        ids = results.boxes.id.cpu().numpy().astype(int)
        confs = results.boxes.conf.cpu().numpy()

        for box, obj_id, conf in zip(boxes, ids, confs):
            if conf > SCORE_THRESH:
                xmin, ymin, xmax, ymax = box
                # --- Generación de color dinámico único basado en el ID ---
                color = (int((obj_id * 50) % 255), 255, int((obj_id * 30) % 255))
                # Dibujar Rectángulo y metadatos (ID y Confianza)
                cv2.rectangle(frame_crop, (xmin, ymin), (xmax, ymax), color, 2)
                display_text = f"ID:{obj_id} Pers: {conf:.2f}"
                cv2.putText(frame_crop, display_text, (xmin, ymin - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # 7. Guardar el cuadro anotado en el video de salida (funciona en local y Colab)
    writer.write(frame_crop)

    # 8. Despliegue en pantalla SOLO en local (en Colab cv2.imshow no funciona)
    if not IN_COLAB:
        cv2.imshow('Real-Time Tracking', frame_crop)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# 9. Liberar recursos
cap.release()
writer.release()
if not IN_COLAB:
    cv2.destroyAllWindows()

print(f"Listo. Video anotado guardado en '{output_path}'.")

Iniciando Tracking en mps (local) con 'vtest.avi'...
Presiona 'q' en la ventana para salir.
Listo. Video anotado guardado en 'salida_tracking.mp4'.


**Reproducir el video con el seguimiento**

### ¿Qué hace este código?

Muestra el video anotado **dentro del notebook**. Es especialmente útil en **Google Colab**, donde no hay ventana de `cv2.imshow`:

1. La celda de tracking guarda el video con el códec **`mp4v`**, que el reproductor HTML5 `<video>` **no sabe decodificar** (por eso a veces se ve el reproductor pero **en negro / "no se ve nada"**).
2. Por eso aquí lo **re-codificamos a H.264** (`salida_tracking_web.mp4`) con **`imageio-ffmpeg`**, que **incluye su propio `ffmpeg`** y funciona en local y en Colab **sin instalar nada del sistema**.
3. Leemos ese archivo, lo codificamos en **base64** y lo incrustamos en un reproductor `<video>` con controles.

> 💡 Si prefieres, también puedes abrir `salida_tracking_web.mp4` directamente con tu reproductor de video.


In [ ]:
# Reproduce el video anotado dentro del notebook (ideal para Colab).
import os
import sys
import subprocess
from base64 import b64encode
from IPython.display import HTML, display

src_path = 'salida_tracking.mp4'        # video que genera la celda de tracking (códec 'mp4v')
web_path = 'salida_tracking_web.mp4'    # copia en H.264, el único que reproduce el navegador


def to_h264(src, dst):
    """Re-codifica a H.264 con imageio-ffmpeg (incluye su propio ffmpeg: sirve en local y Colab)."""
    try:
        import imageio_ffmpeg  # backend que aporta el binario ffmpeg que usa imageio
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio[ffmpeg]"], check=True)
    import imageio.v2 as imageio

    reader = imageio.get_reader(src)
    fps = reader.get_meta_data().get('fps', 25)
    writer = imageio.get_writer(dst, fps=fps, codec='libx264', quality=8, macro_block_size=None)
    for frame in reader:
        writer.append_data(frame)
    reader.close()
    writer.close()


if not os.path.exists(src_path):
    print(f"Aún no existe '{src_path}'. Ejecuta primero la celda de tracking.")
else:
    # El reproductor HTML5 solo entiende H.264; el video original usa 'mp4v', así que lo convertimos.
    try:
        to_h264(src_path, web_path)
        play_path = web_path
    except Exception as e:
        print(f"No se pudo re-codificar a H.264 ({e}); se intentará reproducir el original.")
        play_path = src_path

    mp4 = open(play_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
        <video width=400 controls>
            <source src="{data_url}" type="video/mp4">
        </video>
    '''))
